#### Notebook for running Chain-of-Thought with no supporting context experiments

In [ ]:
import sys, os

sys.path.append("..")
root = "../root/"

In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
from util import summarize_trial, log_trial, save_agents
import joblib
from agents import CoTAgent, ReflexionStrategy

#### Load the HotPotQA Sample

In [ ]:
hotpot = joblib.load("../data/hotpot-qa-distractor-sample.joblib").reset_index(
    drop=True
)
hotpot = hotpot[:2]

hotpot

#### Define the Reflexion Strategy

In [ ]:
print(ReflexionStrategy.__doc__)

In [ ]:
strategy: ReflexionStrategy = ReflexionStrategy.REFLEXION

#### Initialize a CoTAgent for each question

In [ ]:
from prompts import (
    cot_simple_reflect_agent_prompt,
    cot_simple_reflect_prompt,
    cot_simple_agent_prompt,
)
from fewshots import COTQA_SIMPLE6, COT_SIMPLE_REFLECTION

agents = [
    CoTAgent(
        question=row["question"],
        context="",
        key=row["answer"],
        agent_prompt=(
            cot_simple_agent_prompt
            if strategy == ReflexionStrategy.NONE
            else cot_simple_reflect_agent_prompt
        ),
        cot_examples=COTQA_SIMPLE6,
        reflect_prompt=cot_simple_reflect_prompt,
        reflect_examples=COT_SIMPLE_REFLECTION,
    )
    for _, row in hotpot.iterrows()
]

#### Run `n` trials

In [ ]:
n = 2
trial = 0
log = ""

In [ ]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        agent.run(reflexion_strategy=strategy)
        print(f"Answer: {agent.key}\n")
    trial += 1
    log += log_trial(agents, trial)
    correct, incorrect = summarize_trial(agents)
    print(
        f"Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}"
    )
    print("*" * 20)

#### Save the result log

In [ ]:
with open(
    os.path.join(
        root,
        "CoT",
        "no_context",
        strategy.value,
        f"{len(agents)}_questions_{trial}_trials.txt",
    ),
    "w",
) as f:
    f.write(log)
save_agents(agents, os.path.join(root, "CoT", "no_context", strategy.value, "agents"))